# Прогнозирование успеваемости студентов: Pass / Fail

Этот ноутбук содержит один чистый и воспроизводимый pipeline машинного обучения.

**Цель:** предсказать, получит ли студент статус `Fail (0)` или `Pass (1)`, учитывая сильный дисбаланс классов.

## Финальный workflow
1. Загрузить данные и проверить дисбаланс классов.
2. Сравнить baseline и два набора признаков.
3. Выполнить одно стратифицированное разделение Train / Validation / Test.
4. Обучить scaler только на Train.
5. Подобрать параметры Logistic Regression и Linear SVM по Validation.
6. Один раз оценить выбранные модели на Test.
7. Проанализировать confusion matrices, ложные тревоги, пропущенных студентов класса Fail, совпадение предсказаний моделей и probability threshold.
8. Указать ограничения исследования.

> **Важно:** в проекте оставлен только один последовательный pipeline. Старые варианты экспериментов с разделением 80/20 и RBF SVM не используются.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
C_VALUES = [0.01, 0.1, 1, 10]
CLASS_WEIGHTS = [None, "balanced"]

In [ ]:
url = "https://huggingface.co/datasets/jason1966/algozee_student/resolve/main/student_performance_interactions.csv"
df = pd.read_csv(url)

print("Размер датасета:", df.shape)
display(df.head())
print("\nРаспределение целевой переменной:")
display(df["pass_fail"].value_counts().sort_index())
print("\nДоли классов:")
display(df["pass_fail"].value_counts(normalize=True).sort_index())

## 1. Группы признаков и исследовательский вопрос

Вместо того чтобы удалять признаки только потому, что их много, сравниваются два набора:

- **Только академические признаки**
- **Академические + признаки образа жизни**

**Исследовательский вопрос:** помогает ли информация об образе жизни улучшить прогнозирование академического риска?

In [ ]:
academic_features = [
    "previous_score",
    "math_prev_score",
    "science_prev_score",
    "language_prev_score",
    "daily_study_hours",
    "attendance_percentage",
    "homework_completion_rate",
]

lifestyle_features = [
    "sleep_hours",
    "screen_time_hours",
]

all_features = academic_features + lifestyle_features
y = df["pass_fail"]

print("Академические признаки:", academic_features)
print("Признаки образа жизни:", lifestyle_features)
print("Всего выбранных признаков:", len(all_features))

In [ ]:
# Baseline: всегда предсказывать Pass (1)
y_baseline = np.ones(len(y), dtype=int)

print("Baseline — всегда предсказывать Pass")
print("Accuracy:", round(accuracy_score(y, y_baseline), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y, y_baseline), 4))
print("Macro-F1:", round(f1_score(y, y_baseline, average="macro"), 4))
print("Fail Recall:", round(recall_score(y, y_baseline, pos_label=0, zero_division=0), 4))
print("Fail F1:", round(f1_score(y, y_baseline, pos_label=0, zero_division=0), 4))

### Интерпретация baseline

Высокая `Accuracy` сама по себе может быть бесполезной для несбалансированной задачи.

Здесь baseline всегда предсказывает `Pass`, поэтому может получить высокую общую Accuracy, но при этом не находит ни одного студента класса `Fail`. Следовательно, `Fail Recall = 0`.

Именно поэтому для данной задачи необходимо смотреть не только на Accuracy, но и на `Balanced Accuracy`, `Macro-F1`, `Fail Precision`, `Fail Recall` и `Fail F1`.

## 2. Одно стратифицированное разделение Train / Validation / Test

Примерные пропорции: **70% / 15% / 15%**.

- **Train** используется для обучения моделей.
- **Validation** используется для выбора параметров моделей.
- **Test** остаётся отдельным и используется только для финальной оценки.

Test-набор не должен использоваться для выбора `C`, `class_weight` или probability threshold.

In [ ]:
X_all = df[all_features]

X_temp, X_test, y_temp, y_test = train_test_split(
    X_all, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.17647,
    random_state=RANDOM_STATE, stratify=y_temp
)

for name, target in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    print(f"{name}: n={len(target)}, Fail={(target == 0).sum()}, Pass={(target == 1).sum()}")

In [ ]:
# Обучаем scaler ТОЛЬКО на Train, чтобы избежать утечки данных.
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
def validation_metrics(y_true, y_pred):
    return {
        "Macro-F1": f1_score(y_true, y_pred, average="macro"),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Fail Precision": precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        "Fail Recall": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "Fail F1": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
    }

## 3. Подбор гиперпараметров по Validation

Для обеих моделей исследуются:

- `C = 0.01, 0.1, 1, 10`
- `class_weight = None / balanced`

Для сравнения используются метрики, подходящие для несбалансированной задачи.

Для SVM используется только:

`kernel="linear"`

Это соответствует линейной soft-margin SVM, рассматриваемой в проекте.

In [ ]:
# Подбор параметров Logistic Regression
log_rows = []

for C in C_VALUES:
    for weight in CLASS_WEIGHTS:
        model = LogisticRegression(
            C=C,
            class_weight=weight,
            max_iter=1000,
            random_state=RANDOM_STATE
        )
        model.fit(X_train_scaled, y_train)
        row = {"C": C, "class_weight": str(weight)}
        row.update(validation_metrics(y_val, model.predict(X_val_scaled)))
        log_rows.append(row)

log_results = pd.DataFrame(log_rows).sort_values(
    ["Macro-F1", "Balanced Accuracy", "Fail F1"],
    ascending=False
).reset_index(drop=True)

display(log_results)
best_log = log_results.iloc[0]
print("Выбранная Logistic Regression:")
display(best_log.to_frame().T)

In [ ]:
# Подбор параметров Linear soft-margin SVM
svm_rows = []

for C in C_VALUES:
    for weight in CLASS_WEIGHTS:
        model = SVC(
            kernel="linear",
            C=C,
            class_weight=weight
        )
        model.fit(X_train_scaled, y_train)
        row = {"C": C, "class_weight": str(weight)}
        row.update(validation_metrics(y_val, model.predict(X_val_scaled)))
        svm_rows.append(row)

svm_results = pd.DataFrame(svm_rows).sort_values(
    ["Macro-F1", "Balanced Accuracy", "Fail F1"],
    ascending=False
).reset_index(drop=True)

display(svm_results)
best_svm = svm_results.iloc[0]
print("Выбранная Linear SVM:")
display(best_svm.to_frame().T)

## 4. Финальные модели и оценка на Test

Выбранные гиперпараметры фиксируются **до** просмотра результатов на Test-наборе.

После этого модели обучаются на Train и один раз оцениваются на Test.

In [ ]:
final_logistic = LogisticRegression(
    C=float(best_log["C"]),
    class_weight=None if best_log["class_weight"] == "None" else "balanced",
    max_iter=1000,
    random_state=RANDOM_STATE
)

final_svm = SVC(
    kernel="linear",
    C=float(best_svm["C"]),
    class_weight=None if best_svm["class_weight"] == "None" else "balanced"
)

final_logistic.fit(X_train_scaled, y_train)
final_svm.fit(X_train_scaled, y_train)

pred_logistic = final_logistic.predict(X_test_scaled)
pred_svm = final_svm.predict(X_test_scaled)

In [ ]:
def evaluate_model(name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    actual_fail_pred_fail = cm[0, 0]
    actual_fail_pred_pass = cm[0, 1]
    actual_pass_pred_fail = cm[1, 0]
    actual_pass_pred_pass = cm[1, 1]

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Macro-F1": f1_score(y_true, y_pred, average="macro"),
        "Fail Precision": precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        "Fail Recall": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "Fail F1": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        "Фактически Fail → Предсказано Fail": actual_fail_pred_fail,
        "Фактически Fail → Предсказано Pass (пропущен Fail)": actual_fail_pred_pass,
        "Фактически Pass → Предсказано Fail (ложная тревога)": actual_pass_pred_fail,
        "Фактически Pass → Предсказано Pass": actual_pass_pred_pass,
    }

test_results = pd.DataFrame([
    evaluate_model("Logistic Regression", y_test, pred_logistic),
    evaluate_model("Linear SVM", y_test, pred_svm),
])

display(test_results)

In [ ]:
# Confusion matrices с понятными названиями классов вместо использования только TN/FP/FN/TP.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, name, pred in zip(
    axes,
    ["Logistic Regression", "Linear SVM"],
    [pred_logistic, pred_svm]
):
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred,
        labels=[0, 1],
        display_labels=["Fail", "Pass"],
        ax=ax,
        cmap="Blues",
        colorbar=False
    )
    ax.set_title(name)

plt.tight_layout()
plt.show()

### Интерпретация ошибок

Поскольку:

- `0 = Fail`
- `1 = Pass`

для этого проекта удобнее описывать ошибки напрямую, а не только через `TN / FP / FN / TP`.

Наиболее опасная ошибка:

**Actual Fail → Predicted Pass**

Это студент группы риска, которого модель не обнаружила.

Основная ложная тревога:

**Actual Pass → Predicted Fail**

Это студент, которого модель ошибочно отнесла к группе риска.

Поэтому одной высокой `Balanced Accuracy` недостаточно. Необходимо анализировать компромисс между:

- высоким `Fail Recall` — модель пропускает меньше студентов класса Fail;
- высоким `Fail Precision` — меньше студентов Pass ошибочно попадают в группу риска.

Например, если `Fail Recall = 1.0`, но `Fail Precision` низкий, модель может находить всех студентов группы риска, но одновременно создавать большое количество ложных тревог.

In [ ]:
# Проверяем, дали ли Logistic Regression и Linear SVM одинаковые предсказания на Test.
different_predictions = np.sum(pred_logistic != pred_svm)
agreement_rate = np.mean(pred_logistic == pred_svm)

print("Количество объектов Test с разными предсказаниями:", different_predictions)
print("Доля совпадающих предсказаний:", round(agreement_rate, 4))

if different_predictions == 0:
    print("Две модели дали полностью одинаковые предсказания на Test-наборе.")
    print("Для этих масштабированных признаков и выбранных гиперпараметров их линейные границы решений")
    print("дали одинаковые метки классов для каждого объекта Test.")
else:
    comparison = pd.DataFrame({
        "Истинный класс": y_test.to_numpy(),
        "Предсказание Logistic Regression": pred_logistic,
        "Предсказание SVM": pred_svm
    })
    display(comparison[comparison["Предсказание Logistic Regression"] != comparison["Предсказание SVM"]])

## 5. Только академические признаки vs Академические + признаки образа жизни

Для честного сравнения используются одинаковые индексы Train / Validation / Test для обоих наборов признаков.

Сравнение выполняется с помощью Logistic Regression с `class_weight="balanced"` как контролируемый эксперимент.

Таким образом проверяется исследовательский вопрос:

**Добавляет ли информация об образе жизни полезную информацию для прогнозирования академического риска?**

In [ ]:
# Восстанавливаем те же разделения данных с помощью индексов строк.
indices = np.arange(len(df))

idx_temp, idx_test = train_test_split(
    indices, test_size=0.15, random_state=RANDOM_STATE, stratify=y
)
idx_train, idx_val = train_test_split(
    idx_temp, test_size=0.17647, random_state=RANDOM_STATE, stratify=y.iloc[idx_temp]
)

def compare_feature_set(feature_list, name):
    X = df[feature_list]
    scaler_fs = StandardScaler()

    Xtr = scaler_fs.fit_transform(X.iloc[idx_train])
    Xv = scaler_fs.transform(X.iloc[idx_val])
    Xte = scaler_fs.transform(X.iloc[idx_test])

    model = LogisticRegression(
        C=1,
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE
    )
    model.fit(Xtr, y.iloc[idx_train])
    pred = model.predict(Xte)

    row = {"Набор признаков": name, "Количество признаков": len(feature_list)}
    row.update(validation_metrics(y.iloc[idx_test], pred))
    return row

feature_results = pd.DataFrame([
    compare_feature_set(academic_features, "Только академические"),
    compare_feature_set(all_features, "Академические + образ жизни")
])

display(feature_results)

print("Интерпретация: если второй вариант улучшает метрики класса меньшинства,")
print("информация об образе жизни добавляет предсказательную ценность; иначе более простой академический набор может быть предпочтительнее.")

## 6. Эксперимент с probability threshold для Logistic Regression

`class_weight="balanced"` изменяет поведение модели, но также можно изменить порог принятия решения.

Сначала порог выбирается по **Validation set**, а затем его результат оценивается на Test.

- Более низкий threshold → обычно выше `Fail Recall`, но больше ложных тревог.
- Более высокий threshold → обычно выше `Fail Precision`, но больше риск пропустить студентов класса Fail.

Этот эксперимент позволяет явно исследовать компромисс между `Fail Precision` и `Fail Recall`, а не просто принимать стандартное правило классификации как окончательное решение.

In [ ]:
# Вероятность Fail находится в столбце, соответствующем классу 0.
val_fail_proba = final_logistic.predict_proba(X_val_scaled)[:, list(final_logistic.classes_).index(0)]

thresholds = np.arange(0.05, 0.96, 0.05)
threshold_rows = []

for threshold in thresholds:
    # Предсказываем Fail (0), когда P(Fail) >= threshold.
    val_pred = np.where(val_fail_proba >= threshold, 0, 1)
    row = {"Threshold": round(float(threshold), 2)}
    row.update(validation_metrics(y_val, val_pred))
    threshold_rows.append(row)

threshold_results = pd.DataFrame(threshold_rows)
display(threshold_results)

# Choose threshold by best Validation Fail F1, then Balanced Accuracy.
best_threshold_row = threshold_results.sort_values(
    ["Fail F1", "Balanced Accuracy", "Fail Recall"],
    ascending=False
).iloc[0]

best_threshold = float(best_threshold_row["Threshold"])
print("Выбранный threshold по Validation:", best_threshold)
display(best_threshold_row.to_frame().T)

In [ ]:
test_fail_proba = final_logistic.predict_proba(X_test_scaled)[:, list(final_logistic.classes_).index(0)]

threshold_pred = np.where(test_fail_proba >= best_threshold, 0, 1)

threshold_test_result = pd.DataFrame([
    evaluate_model(
        f"Logistic Regression, threshold={best_threshold:.2f}",
        y_test,
        threshold_pred
    )
])

display(threshold_test_result)

print("Сравните эту строку с результатом модели при стандартном правиле классификации выше.")
print("Эксперимент с threshold явно показывает компромисс между Precision и Recall, а не принимает одно стандартное правило решения как окончательное.")

## 7. Итоговые выводы и ограничения

### Основные выводы

- Общая `Accuracy` может быть вводящей в заблуждение для несбалансированного датасета.
- Baseline `Always predict Pass` может иметь высокую Accuracy, но при этом находить **0 студентов класса Fail**.
- Модели с balancing могут снизить общую Accuracy, но значительно улучшить `Fail Recall`.
- Высокий `Fail Recall` при низком `Fail Precision` означает, что модель находит больше студентов группы риска, но одновременно создаёт много ложных тревог.
- Logistic Regression и Linear SVM необходимо сравнивать не только по итоговым метрикам, но и по количеству объектов, для которых они дают разные предсказания.
- Сравнение `Academic only` и `Academic + lifestyle` превращает feature selection в полноценный исследовательский эксперимент.
- Probability threshold даёт дополнительный способ регулировать компромисс между пропущенными студентами класса Fail и ложными тревогами.

### Ограничения проекта

- Во всём датасете около 70 примеров класса Fail.
- После разделения данных в Validation и Test остаётся примерно по 10–11 студентов класса Fail.
- Поэтому выбор гиперпараметров на одном Validation split может быть нестабильным.
- Более сильная версия проекта должна использовать **Stratified 5-Fold Cross-Validation** для выбора гиперпараметров, после чего должна выполняться одна финальная оценка на отдельном Test-наборе.
- Результаты относятся к данному конкретному датасету и выбранным признакам. Без внешней проверки их нельзя автоматически переносить на другие данные.